In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

tokenizer = AutoTokenizer.from_pretrained("gpt2")

def tokenize(example):
    return tokenizer(example["text"])

dataset = dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [2]:
block_size = 32

def group_texts(examples):
    tokens = sum(examples["input_ids"], [])
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]

    result = {
        "input_ids": [tokens[i:i+block_size] for i in range(0, total_length, block_size)]
    }

    result["labels"] = result["input_ids"].copy()
    return result

dataset = dataset.map(group_texts, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "labels"])

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

ArrowInvalid: Column 1 named input_ids expected length 1000 but got length 2201

In [5]:
import torch.nn as nn

class positional_encoding(nn.Module):
    def __init__(self, max_len, dim):
        super().__init__()
        self.pos = nn.Embedding(max_len, dim)
    
    def forward(self, x):
        B, T = x.shape
        positions = x.arange(T, device=x.device)
        return self.pos(positions)